# Model 4 — Transformer Encoder

**Efficiency strategy:** Preload the full normalized (N, 301, 3) dataset into GPU VRAM once. Batch size raised to 256 (was capped at 128 with HDF5 loading, now safe with preloaded data and attention at ~354 MiB per layer).

**Architecture:**
```
Input (batch, 301, 3)
Linear(3 → 64)                  input projection
nn.Embedding(301, 64)           learnable positional encoding
TransformerEncoder(
    d_model=64, nhead=4, ffn=256, dropout=0.1,
    num_layers=3
)
Mean pool over 301 positions → (batch, 64)
Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1)
```

**Data:** `data/processed/sequences.h5` — battery-level split: 22 train / 6 test batteries.

In [ ]:
import h5py
import json
import math
import os
import sys
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import BatterySOHDataset, DEFAULT_TEST_BATTERIES
from src.models.transformer import SOHTransformer

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
H5_PATH      = "../data/processed/sequences.h5"
STATS_PATH   = "../data/processed/norm_stats.json"
CKPT_PATH    = "../results/checkpoints/transformer_best.pt"
METRICS_PATH = "../results/metrics/transformer_history.json"
FIG_DIR      = "../results/figures"

EPOCHS       = 100
BATCH_SIZE   = 256   # was 128; safe with preloaded VRAM data (attn ~354 MiB, 6 GB headroom)
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
RANDOM_SEED  = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  Batch size: {BATCH_SIZE}")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)

def set_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds()

## Section 1 — Data Loading + GPU Preload

In [ ]:
train_idx, test_idx = BatterySOHDataset.create_splits(H5_PATH)

if not os.path.exists(STATS_PATH):
    tmp = BatterySOHDataset(H5_PATH, indices=train_idx, normalize=False)
    tmp.compute_normalization_stats(STATS_PATH)

with open(STATS_PATH) as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

print('Loading dataset into RAM...', flush=True)
with h5py.File(H5_PATH, 'r') as f:
    X_all = f['X'][:]
    y_all = f['y'][:] / 100.0
print(f'  X: {X_all.shape}  ({X_all.nbytes / 1024**3:.2f} GB)')

X_all = (X_all - mean) / (std + 1e-8)

print('Pushing to GPU...', end=' ', flush=True)
X_tensor = torch.from_numpy(X_all).to(DEVICE)
y_tensor = torch.from_numpy(y_all).to(DEVICE)
del X_all, y_all
torch.cuda.empty_cache()
print(f'done. VRAM: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

train_idx_t = torch.from_numpy(train_idx)
test_idx_t  = torch.from_numpy(test_idx)

X_train = X_tensor[train_idx_t]
y_train = y_tensor[train_idx_t]
X_test  = X_tensor[test_idx_t]
y_test  = y_tensor[test_idx_t]
del X_tensor, y_tensor

print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Test batteries: {DEFAULT_TEST_BATTERIES}')
print(f'VRAM after split: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

attn_mb = BATCH_SIZE * 4 * 301 * 301 * 4 / 1e6
print(f'Attention matrices per layer @ batch={BATCH_SIZE}: {attn_mb:.0f} MB')

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f'Train batches/epoch: {len(train_loader)}  (was ~3515 with batch=128 + HDF5)')

## Section 2 — Model Architecture

In [ ]:
set_seeds()
model = SOHTransformer().to(DEVICE)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

dummy = torch.randn(4, 301, 3).to(DEVICE)
with torch.no_grad():
    z = model.input_proj(dummy)
    print(f"\nInput:                 {tuple(dummy.shape)}")
    print(f"After input_proj:      {tuple(z.shape)}")
    z = z + model.pos_embedding(model.positions[:, :301])
    print(f"After pos_embedding:   {tuple(z.shape)}")
    z = model.transformer(z)
    print(f"After transformer:     {tuple(z.shape)}")
    z = z.mean(dim=1)
    print(f"After mean pool:       {tuple(z.shape)}")
    out = model(dummy)
    print(f"Output:                {tuple(out.shape)}")

## Section 3 — Positional Embedding Visualisation

Unlike sinusoidal embeddings, the learnable positional embeddings are random before training. This cell checks their similarity structure — after training, nearby positions should have more similar embeddings.

In [ ]:
with torch.no_grad():
    pos_emb = model.pos_embedding.weight.cpu().numpy()   # (301, 64)

step = 10
idx = list(range(0, 301, step))
sub = pos_emb[idx]
norms = np.linalg.norm(sub, axis=1, keepdims=True)
sim = (sub @ sub.T) / (norms @ norms.T + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(sim, aspect="auto", cmap="Blues", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(idx))); axes[0].set_xticklabels(idx, rotation=90, fontsize=7)
axes[0].set_yticks(range(len(idx))); axes[0].set_yticklabels(idx, fontsize=7)
axes[0].set_title("Positional Embedding Cosine Similarity\n(every 10th position, before training)")
plt.colorbar(im, ax=axes[0])

axes[1].plot(np.linalg.norm(pos_emb, axis=1), color="#C44E52", lw=1.5)
axes[1].set_xlabel("Position (timestep)")
axes[1].set_ylabel("L2 norm")
axes[1].set_title("Positional Embedding L2 Norms (before training)")
axes[1].grid(alpha=0.3)

fig.suptitle("Learnable Positional Embeddings — Initial State", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_pos_enc.png"), dpi=150)
plt.show()

## Section 4 — Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    bar = tqdm(loader, desc='Training', leave=True, unit='batch')
    for x, y in bar:
        # Data already on GPU
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x).squeeze(1), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    bar = tqdm(loader, desc='Validating', leave=True, unit='batch')
    for x, y in bar:
        preds.append(model(x).squeeze(1).cpu())
        targets.append(y.cpu())
    preds   = torch.cat(preds).numpy() * 100.0
    targets = torch.cat(targets).numpy() * 100.0
    mae  = float(np.abs(preds - targets).mean())
    rmse = float(math.sqrt(((preds - targets)**2).mean()))
    return mae, rmse, preds, targets

In [ ]:
set_seeds()
model = SOHTransformer().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
criterion = nn.MSELoss()

history    = {"train_loss": [], "val_mae": [], "val_rmse": []}
best_val_mae = float("inf")
no_improve   = 0

print(f'Batch size: {BATCH_SIZE}  |  Train batches/epoch: {len(train_loader)}')
print(f'VRAM before training: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*60}')

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_mae, val_rmse, _, _ = evaluate(model, test_loader)
    scheduler.step(val_mae)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)

    lr_now = optimizer.param_groups[0]['lr']
    vram   = torch.cuda.memory_allocated() / 1024**2

    print(f'\nTrain Loss: {train_loss:.4f}')
    print(f'Val MAE:    {val_mae:.4f}%')
    print(f'Val RMSE:   {val_rmse:.4f}%')
    print(f'LR:         {lr_now:.6f}')
    print(f'VRAM:       {vram:.0f} MiB')

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        no_improve   = 0
        torch.save({"epoch": epoch, "val_mae": val_mae,
                    "model_state_dict": model.state_dict()}, CKPT_PATH)
        print(f'Model saved to {CKPT_PATH}')
        print('✓ New best model saved!')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

with open(METRICS_PATH, "w") as f:
    json.dump(history, f, indent=2)
print(f'\nBest val MAE: {best_val_mae:.4f}%  |  Checkpoint: {CKPT_PATH}')

## Section 5 — Results

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val MAE {ckpt['val_mae']:.4f}%)")

test_mae, test_rmse, preds, targets = evaluate(model, test_loader)
print(f"\nTest MAE:  {test_mae:.4f} SOH%")
print(f"Test RMSE: {test_rmse:.4f} SOH%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], color="#C44E52", lw=1.5)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_mae"],  label="Val MAE",  color="darkorange", lw=1.5)
axes[1].plot(history["val_rmse"], label="Val RMSE", color="crimson",    lw=1.5, ls="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("SOH%")
axes[1].set_title("Validation Metrics"); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle("Transformer Encoder — Training Curves", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_training_curves.png"), dpi=150)
plt.show()

In [ ]:
errors = preds - targets
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lims = [min(targets.min(), preds.min()) - 2, max(targets.max(), preds.max()) + 2]
axes[0].scatter(targets, preds, alpha=0.25, s=5, color="#C44E52")
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Actual SOH (%)"); axes[0].set_ylabel("Predicted SOH (%)")
axes[0].set_title(f"Predicted vs Actual  MAE={test_mae:.2f}%  RMSE={test_rmse:.2f}%")
axes[0].grid(alpha=0.3)

axes[1].hist(errors, bins=60, color="#C44E52", edgecolor="white", alpha=0.85)
axes[1].axvline(0,              color="black", lw=1.2, ls="--", label="Zero error")
axes[1].axvline(errors.mean(),  color="red",   lw=1.5, label=f"Mean={errors.mean():.2f}%")
axes[1].set_xlabel("Prediction Error (SOH%)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Error Distribution  std={errors.std():.2f}%")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Transformer Encoder — Test Set Results", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_results.png"), dpi=150)
plt.show()

## Section 6 — Attention Visualisation

Which timesteps does the model focus on when predicting SOH? We average attention weights across heads and samples to get a per-layer "attention profile" over the 301 sequence positions.

In [ ]:
n_viz = min(16, BATCH_SIZE)
attn_weights = model.get_attention_weights(X_test[:n_viz])
# Each element: (n_viz, n_heads, 301, 301)

n_layers = len(attn_weights)
fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5))
if n_layers == 1:
    axes = [axes]

for i, (ax, attn) in enumerate(zip(axes, attn_weights)):
    mean_attn = attn.mean(dim=(0, 1)).numpy()
    im = ax.imshow(mean_attn, aspect="auto", cmap="Blues", interpolation="nearest")
    ax.set_title(f"Layer {i+1}", fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"Self-Attention Weights (mean over {n_viz} samples, all heads)",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_attn_heatmap.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 3), sharey=True)
if n_layers == 1:
    axes = [axes]

for i, (ax, attn) in enumerate(zip(axes, attn_weights)):
    profile = attn.mean(dim=(0, 1)).sum(dim=0).numpy()
    ax.plot(profile, color="#C44E52", lw=1.2)
    ax.set_xlabel("Timestep position")
    if i == 0:
        ax.set_ylabel("Total attention received")
    ax.set_title(f"Layer {i+1} — Attention Profile")
    ax.grid(alpha=0.3)

fig.suptitle("Which Timesteps Receive the Most Attention?", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_attn_profile.png"), dpi=150)
plt.show()

In [ ]:
with torch.no_grad():
    pos_emb_trained = model.pos_embedding.weight.cpu().numpy()

step = 10
idx = list(range(0, 301, step))
sub = pos_emb_trained[idx]
norms = np.linalg.norm(sub, axis=1, keepdims=True)
sim = (sub @ sub.T) / (norms @ norms.T + 1e-8)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(sim, aspect="auto", cmap="Blues", vmin=-1, vmax=1)
ax.set_xticks(range(len(idx))); ax.set_xticklabels(idx, rotation=90, fontsize=7)
ax.set_yticks(range(len(idx))); ax.set_yticklabels(idx, fontsize=7)
ax.set_title("Positional Embedding Cosine Similarity — After Training")
plt.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transformer_pos_enc_trained.png"), dpi=150)
plt.show()

In [ ]:
print("=" * 44)
print("TRANSFORMER ENCODER — FINAL SUMMARY")
print("=" * 44)
print(f"Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"d_model    : 64  |  heads: 4  |  layers: 3")
print(f"Batch size : {BATCH_SIZE}  (batches/epoch: {len(train_loader)})")
print(f"Test MAE   : {test_mae:.4f} SOH%")
print(f"Test RMSE  : {test_rmse:.4f} SOH%")
print(f"Error mean : {errors.mean():.4f} SOH%")
print(f"Error std  : {errors.std():.4f} SOH%")
print(f"Error p95  : {np.percentile(np.abs(errors), 95):.4f} SOH%")